In [1]:
import sqlite3
import pandas as pd
import os

### 데이터베이스 생성

In [2]:
conn = sqlite3.connect('olist.db')

In [3]:
csv_files = [
    'data/customers.csv',
    'data/orders.csv',
    'data/order_items.csv',
    'data/products.csv',
    'data/sellers.csv',
    'data/order_payments.csv',
    'data/order_reviews.csv',
    'data/geolocation.csv',
    'data/product_category_name_translation.csv'
]

for file in csv_files:
    table_name = os.path.splitext(os.path.basename(file))[0]
    df = pd.read_csv(file)
    df.to_sql(table_name, conn, if_exists='replace', index=False)

### 테이블 추출

In [4]:
query = """
SELECT *
FROM customers
"""
customers=pd.read_sql(query, conn)
customers.head(5)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [5]:
query = """
SELECT *
FROM orders
"""
orders=pd.read_sql(query, conn)
orders.head(5)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [6]:
query = """
SELECT *
FROM order_reviews
"""
reviews=pd.read_sql(query, conn)
reviews.head(5)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,None,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,None,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


### 첫 주문 배송 지연 여부와 재구매율

In [7]:
# order_status = 'delivered'  -> 구매만 하고 취소된 주문 제외
delivered = orders[orders.order_status == 'delivered'].copy()
# 예상배송일 보다 실제배송일이 더 지연된 주문을 배송지연으로 정의
delivered['is_delayed'] = delivered['order_delivered_customer_date'] > delivered['order_estimated_delivery_date']
# 실제 배송일 정보가 없는 데이터 8개는 배송지연여부 판단이 불가하여 제외
delivered = delivered.merge(customers.loc[:,['customer_id','customer_unique_id']],on='customer_id')\
[['customer_unique_id','order_purchase_timestamp','order_delivered_customer_date','order_estimated_delivery_date','is_delayed']].dropna()
delivered

,customer_unique_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,is_delayed
0,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18 00:00:00,False
1,af07308b275d755c9edb36a90c618231,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13 00:00:00,False
2,3a653a41f6f9fc3d2a113cf8398680e8,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04 00:00:00,False
3,7c142cf63193a1473d2e66489a9ae977,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15 00:00:00,False
4,72632f0f9dd73dfee390c9b22eb56dd6,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26 00:00:00,False
...,...,...,...,...,...
96473,6359f309b166b0196dbf7ad2ac62bb5a,2017-03-09 09:54:05,2017-03-17 15:08:01,2017-03-28 00:00:00,False
96474,da62f9e57a76d978d02ab5362c509660,2018-02-06 12:58:58,2018-02-28 17:37:56,2018-03-02 00:00:00,False
96475,737520a9aad80b3fbbdad19b66b37b30,2017-08-27 14:46:43,2017-09-21 11:24:17,2017-09-27 00:00:00,False
96476,5097a5312c8b157bb7be58ae360ef43c,2018-01-08 21:28:27,2018-01-25 23:32:54,2018-02-15 00:00:00,False


In [8]:
# customer_unique_id별 order_purchase_timestamp가 가장 빠른 주문을 첫 주문으로 정의
first_orders = delivered.sort_values('order_purchase_timestamp').drop_duplicates('customer_unique_id',keep='first')\
.drop(['order_delivered_customer_date','order_estimated_delivery_date'],axis=1).rename({'order_purchase_timestamp':'first_order_date'},axis=1)
first_orders

,customer_unique_id,first_order_date,is_delayed
29811,830d5b7aaa3b6f1e9ad63703bec97d23,2016-09-15 12:16:38,True
90510,32ea3bdedab835c3aa6cb68ce66565ef,2016-10-03 09:44:50,False
27599,2f64e403852e6893ae37485d5fcacdaf,2016-10-03 16:56:50,False
95059,61db744d2f835035a5625b59350c6b63,2016-10-03 21:13:36,False
85830,8d3a54507421dbd2ce0a1d58046826e0,2016-10-03 22:06:03,False
...,...,...,...
96407,7a22d14aa3c3599238509ddca4b93b01,2018-08-29 12:25:59,False
29196,5c58de6fb80e93396e2f35642666b693,2018-08-29 14:18:23,False
30554,7febafa06d9d8f232a900a2937f04338,2018-08-29 14:18:28,False
67603,b701bebbdf478f5500348f03aff62121,2018-08-29 14:52:00,False


In [9]:
# customer_unique_id가 두개 이상, order_purchase_timestamp가 1일 이상 60일 이내인 주문을 재구매로 정의
repurchase_orders = delivered.merge(first_orders[['customer_unique_id','first_order_date']],on='customer_unique_id').query('first_order_date < order_purchase_timestamp')
repurchase_orders['days_from_first'] = (pd.to_datetime(repurchase_orders['order_purchase_timestamp']) - pd.to_datetime(repurchase_orders['first_order_date'])).dt.days
repurchase_orders = repurchase_orders.query('(days_from_first >=1) & (days_from_first <= 60)')
repurchase_orders

,customer_unique_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,is_delayed,first_order_date,days_from_first
0,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18 00:00:00,False,2017-09-04 11:26:38,27
70,fa0ee7ceb94193fb02aa78ce3a55695a,2018-02-15 10:33:30,2018-02-24 19:15:56,2018-03-07 00:00:00,False,2018-01-01 20:49:02,44
144,65ee274b862c5c053367be8a70dbc029,2018-05-25 08:54:21,2018-05-30 15:06:44,2018-06-12 00:00:00,False,2018-04-25 10:49:37,29
318,cef44ce121b124fdc3438472b1ac4e9a,2018-02-09 13:48:17,2018-02-22 17:14:00,2018-03-06 00:00:00,False,2018-01-08 21:43:53,31
394,5c00e849b56a56ea31560d5d66f933e9,2018-08-12 12:04:03,2018-08-22 17:09:01,2018-08-30 00:00:00,False,2018-07-04 09:53:34,39
...,...,...,...,...,...,...,...
96060,b530945168f12f82a3f9be3a5e243afa,2017-10-13 22:52:40,2017-10-24 18:18:42,2017-10-31 00:00:00,False,2017-08-19 11:09:55,55
96115,1bcbeccb2851888f299cf9a5dba8553d,2018-08-05 21:54:15,2018-08-10 18:20:37,2018-08-30 00:00:00,False,2018-06-10 21:27:49,56
96208,4c4b5dc90eb4e33188153ca3e7999f92,2018-04-08 13:05:01,2018-04-13 11:32:05,2018-04-20 00:00:00,False,2018-04-06 16:47:38,1
96259,f0a8e36658d6d08f6e96b9ecf389fd52,2018-06-09 21:51:46,2018-06-14 00:21:19,2018-06-26 00:00:00,False,2018-05-03 22:16:54,36


In [10]:
repurchase_orders['customer_unique_id'].drop_duplicates()

0        7c396fd4830fd04220f754e42b4e5bff
70       fa0ee7ceb94193fb02aa78ce3a55695a
144      65ee274b862c5c053367be8a70dbc029
318      cef44ce121b124fdc3438472b1ac4e9a
394      5c00e849b56a56ea31560d5d66f933e9
                       ...               
96060    b530945168f12f82a3f9be3a5e243afa
96115    1bcbeccb2851888f299cf9a5dba8553d
96208    4c4b5dc90eb4e33188153ca3e7999f92
96259    f0a8e36658d6d08f6e96b9ecf389fd52
96328    e2492e4188991b6276a4a62a287a5451
Name: customer_unique_id, Length: 889, dtype: object

In [11]:
df = first_orders.assign(is_repurchased= lambda df: df['customer_unique_id'].isin(repurchase_orders['customer_unique_id'].drop_duplicates()))
df

,customer_unique_id,first_order_date,is_delayed,is_repurchased
29811,830d5b7aaa3b6f1e9ad63703bec97d23,2016-09-15 12:16:38,True,False
90510,32ea3bdedab835c3aa6cb68ce66565ef,2016-10-03 09:44:50,False,False
27599,2f64e403852e6893ae37485d5fcacdaf,2016-10-03 16:56:50,False,False
95059,61db744d2f835035a5625b59350c6b63,2016-10-03 21:13:36,False,False
85830,8d3a54507421dbd2ce0a1d58046826e0,2016-10-03 22:06:03,False,False
...,...,...,...,...
96407,7a22d14aa3c3599238509ddca4b93b01,2018-08-29 12:25:59,False,False
29196,5c58de6fb80e93396e2f35642666b693,2018-08-29 14:18:23,False,False
30554,7febafa06d9d8f232a900a2937f04338,2018-08-29 14:18:28,False,False
67603,b701bebbdf478f5500348f03aff62121,2018-08-29 14:52:00,False,False


In [12]:
df.groupby('is_delayed')['is_repurchased'].agg(customer_cnt='count',repurchase_rate='mean').reset_index()

,is_delayed,customer_cnt,repurchase_rate
0,False,85748,0.009656
1,True,7602,0.008024


- 첫 주문 배송이 지연된 고객의 60일 이내 재구매율은 정상 배송 고객 대비 약 0.16% 낮게 나타났으며, 이는 상대적으로 약 17% 낮은 수준임을 확인
- 다만 전체 재구매율이 매우 낮아 배송 개선만으로 재구매율을 크게 개선하기에는 한계가 있음

### 첫 주문 리뷰 점수와 재구매까지 걸린 시간

In [13]:
reviews.review_id.is_unique

False

In [14]:
reviews.order_id.is_unique

False

In [15]:
# order_id별로 리뷰가 여러 개 있을 수 있으므로 리뷰 점수는 평균으로 집계
review_by_order = reviews.groupby('order_id').agg({'review_score':'mean'}).reset_index().eval('review_score=review_score.round(1)').rename({'review_score':'avg_review_score'},axis=1)
review_by_order

,order_id,avg_review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0
...,...,...
98668,fffc94f6ce00a00581880bf54a75a037,5.0
98669,fffcd46ef2263f404302a634eb57f7eb,5.0
98670,fffce4705a9662cd70adb13d4a31832d,5.0
98671,fffe18544ffabc95dfada21779c9644f,5.0


In [16]:
delivered = orders[orders.order_status == 'delivered'].copy()
delivered.merge(customers.loc[:,['customer_id','customer_unique_id']],on='customer_id')

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6
...,...,...,...,...,...,...,...,...,...
96473,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00,6359f309b166b0196dbf7ad2ac62bb5a
96474,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00,da62f9e57a76d978d02ab5362c509660
96475,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00,737520a9aad80b3fbbdad19b66b37b30
96476,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00,5097a5312c8b157bb7be58ae360ef43c


In [17]:
orders_with_review = delivered.merge(customers.loc[:,['customer_id','customer_unique_id']],on='customer_id')\
[['order_id','customer_unique_id','order_purchase_timestamp']].merge(review_by_order,on='order_id',how='left')
orders_with_review

,order_id,customer_unique_id,order_purchase_timestamp,avg_review_score
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:33,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,2018-07-24 20:41:37,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,2018-08-08 08:38:49,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,2017-11-18 19:28:06,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,2018-02-13 21:18:39,5.0
...,...,...,...,...
96473,9c5dedf39a927c1b2549525ed64a053c,6359f309b166b0196dbf7ad2ac62bb5a,2017-03-09 09:54:05,5.0
96474,63943bddc261676b46f01ca7ac2f7bd8,da62f9e57a76d978d02ab5362c509660,2018-02-06 12:58:58,4.0
96475,83c1379a015df1e13d02aae0204711ab,737520a9aad80b3fbbdad19b66b37b30,2017-08-27 14:46:43,5.0
96476,11c177c8e97725db2631073c19f07b62,5097a5312c8b157bb7be58ae360ef43c,2018-01-08 21:28:27,2.0


In [18]:
first_orders = orders_with_review.sort_values('order_purchase_timestamp').drop_duplicates('customer_unique_id',keep='first')\
.rename({'order_purchase_timestamp':'first_order_date'},axis=1).drop('order_id',axis=1)
first_orders

,customer_unique_id,first_order_date,avg_review_score
29811,830d5b7aaa3b6f1e9ad63703bec97d23,2016-09-15 12:16:38,1.0
90510,32ea3bdedab835c3aa6cb68ce66565ef,2016-10-03 09:44:50,4.0
27599,2f64e403852e6893ae37485d5fcacdaf,2016-10-03 16:56:50,4.0
95059,61db744d2f835035a5625b59350c6b63,2016-10-03 21:13:36,3.0
85830,8d3a54507421dbd2ce0a1d58046826e0,2016-10-03 22:06:03,1.0
...,...,...,...
96407,7a22d14aa3c3599238509ddca4b93b01,2018-08-29 12:25:59,1.0
29196,5c58de6fb80e93396e2f35642666b693,2018-08-29 14:18:23,5.0
30554,7febafa06d9d8f232a900a2937f04338,2018-08-29 14:18:28,5.0
67603,b701bebbdf478f5500348f03aff62121,2018-08-29 14:52:00,3.0


In [19]:
repurchase_orders = orders_with_review.merge(first_orders[['customer_unique_id','first_order_date']],on='customer_unique_id').query('first_order_date < order_purchase_timestamp')
repurchase_orders['days_from_first'] = (pd.to_datetime(repurchase_orders['order_purchase_timestamp']) - pd.to_datetime(repurchase_orders['first_order_date'])).dt.days
repurchase_orders

,order_id,customer_unique_id,order_purchase_timestamp,avg_review_score,first_order_date,days_from_first
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:33,4.0,2017-09-04 11:26:38,27
14,dcb36b511fcac050b97cd5c05de84dc3,ccafc1c3f270410521c3c6f3b249870f,2018-06-07 19:03:12,5.0,2016-10-06 19:33:34,608
15,403b97836b0c04a622354cf531062e5f,6e26bbeaa107ec34112c64e1ee31c0f5,2018-01-02 19:00:43,NaN,2017-07-04 21:57:51,181
70,6abaad69b8b349c3a529b4b91ce18e46,fa0ee7ceb94193fb02aa78ce3a55695a,2018-02-15 10:33:30,4.0,2018-01-01 20:49:02,44
89,d3d6788577c9592da441752e8a1dd5e3,7973a6ba9c81ecaeb3d628c33c7c7c48,2017-09-19 22:17:15,5.0,2017-09-19 22:17:14,0
...,...,...,...,...,...,...
96266,f0dd9af88d8ef5a8e4670fbbedaf19c4,1d627d8a6e1e33ea8eeeb7a17d998660,2017-09-02 20:38:29,5.0,2017-09-02 20:38:28,0
96267,843f956709f0a664d816bee124544122,f0a8e36658d6d08f6e96b9ecf389fd52,2018-06-09 21:51:46,4.0,2018-05-03 22:16:54,36
96336,a96157730ca02d9de4c4e4ac2fc49f2c,e2492e4188991b6276a4a62a287a5451,2018-02-06 08:38:54,1.0,2018-02-01 08:52:15,4
96384,bfecb4ee6ab98bff69307aab578db48a,dca9a13536adcef18c6c5859487347b1,2018-03-17 12:52:37,5.0,2017-12-26 19:20:48,80


In [20]:
second_orders = repurchase_orders.sort_values('days_from_first').drop_duplicates('customer_unique_id',keep='first')[['customer_unique_id','days_from_first']]
second_orders

,customer_unique_id,days_from_first
34792,e8569c57efe3adf347b3e92ec04d1f91,0
48461,0a61b571f594b6919601bcf3380da7f7,0
86285,518199e5f36305bffce18e38aa2642aa,0
39048,54b716ecf36f372ca6a585a68025e8a1,0
16050,fc1c735fd279ad7321e7b40a4f9d361f,0
...,...,...
54566,a1c61f8566347ec44ea37d22854634a1,524
71274,87b3f231705783eb2217e25851c0a45d,572
65241,94e5ea5a8c1bf546db2739673060c43f,580
40058,d8f3c4f441a9b59a29f977df16724f38,582


In [21]:
df = first_orders.merge(second_orders,on='customer_unique_id')
df

,customer_unique_id,first_order_date,avg_review_score,days_from_first
0,32ea3bdedab835c3aa6cb68ce66565ef,2016-10-03 09:44:50,4.0,358
1,0b3dc7efaafb0cf78a4796d42fa8d74c,2016-10-04 14:49:13,5.0,296
2,4e23e1826902ec9f208e8cc61329b494,2016-10-05 12:32:55,5.0,524
3,94e5ea5a8c1bf546db2739673060c43f,2016-10-05 21:10:56,5.0,580
4,f7b62c75467e8ce080b201667cbbc274,2016-10-06 14:59:56,1.0,0
...,...,...,...,...
2560,53cc7338a7d5994fe8baa28a039d2343,2018-08-15 16:58:08,3.0,0
2561,683459cab0129375a071efb00da7c191,2018-08-17 11:12:26,4.0,8
2562,931a4a1a3e2cf8b4b4d33922f1469dbe,2018-08-17 23:14:16,5.0,0
2563,059e7585d8fcd2430a33375bdbcbb990,2018-08-19 11:55:56,5.0,0


In [22]:
result = df.dropna().assign(review_group = lambda df:['low' if x<=2 else 'high' if x>=4 else 'mid' for x in df.avg_review_score]).query("review_group != 'mid'")\
[['review_group','days_from_first']]
result

,review_group,days_from_first
0,high,358
1,high,296
2,high,524
3,high,580
4,low,0
...,...,...
2559,high,0
2561,high,8
2562,high,0
2563,high,0


In [23]:
result.groupby('review_group')['days_from_first'].agg(['count','mean','median']).reset_index()

,review_group,count,mean,median
0,high,2039,93.493870,45.0
1,low,292,70.719178,22.0


- 첫 주문 리뷰 점수가 낮은 고객군에서 재구매까지 걸린 시간이 더 짧게 나타남